In [11]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "schmid2017great")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "pone.0187451.s002.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [12]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)



In [13]:
data_list = df.values.tolist()
column_name = df.columns.values.tolist()
output = []

for line in data_list: 
    for value,name in zip(line[3:],column_name[3:]): 
        output.append([line[0], line[2], value, name]) 

In [14]:
df = pd.DataFrame(output, columns=['participant', 'order', 'value', 'temp'])
df['study_id']="schmid2017great"

In [15]:
df[['session','trial', 'experiment_name']] = df['temp'].str.split('_',expand=True)
df['session'].replace('session', '', inplace=True, regex=True)
df['trial'].replace('trial', '', inplace=True, regex=True)

In [16]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left')

In [17]:
df['trial']=df['trial'].astype(int)
df['session']=df['session'].astype(int)
df=df.sort_values(by = ['participant','session','trial'])

In [18]:
gbb = df[~df.experiment_name.str.contains("automaticgaze")]
ag = df[~df.experiment_name.str.contains("gazebehindbarriers")]

g_list =gbb[['study_id', 'participant', 'sex', 'species', 
                        'session', 'trial','order', 'value']].values.tolist()

a_list = ag[[ 'value']].values.tolist()
print(len(g_list))
print(len(a_list))


1056
1056


In [19]:

combined_lol = [lol_1+lol_2 for lol_1,lol_2 in zip(g_list,a_list)]
# print(combined_lol) 

df = pd.DataFrame(combined_lol, columns=['study_id', 'participant', 'sex', 'species', 
            'session', 'trial','order','gaze_behind_barriers_value', 'automatic_gaze_value'])

In [20]:
studyID_standardized=df[['study_id', 'participant', 'sex', 'species', 
            'session', 'trial','order', 'automatic_gaze_value','gaze_behind_barriers_value']]
comp_out_path_stand = os.path.join(out_pathway, 'schmid2017great_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'schmid2017great_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)